In [39]:
#Environment Setup

import os
from dotenv import load_dotenv, find_dotenv
import json
from datetime import datetime
from typing import List, Dict, Any, Optional


import ast #Abstract Syntax Tree --  to structurize codes
import sys
from io import StringIO
import contextlib


#load env variables
load_dotenv()
if not os.getenv('GROQ_API_KEY'):
    load_dotenv(find_dotenv(usecwd=True))

# Configuration
GROQ_API_KEY = os.getenv('GROQ_API_KEY')
if not GROQ_API_KEY:
    print("⚠️  Warning: GROQ_API_KEY not found in .env file")
    print("Please add your Groq API key to the .env file")
else:
    print("✅ Groq API key loaded successfully")

print("🚀 Environment setup complete!")



#LLM Integration -- GROQ

import requests

class GroqLLM:
    """Core LLM Integration for the coding assistant"""

    def __init__(self, api_key: str):
        self.api_key = api_key
        self.base_url = "https://api.groq.com/openai/v1/chat/completions"
        self.headers = {
            "Authorization": f"Bearer {api_key}",
            "Content-Type": "application/json"
        }

    def generate_response(self, messages: List[Dict], temperature: float = 0.1) -> str:
        """Generate response from Groq LLM"""

        try:
            payload = {
                "model": "openai/gpt-oss-20b",
                "messages": messages,
                "temperature": temperature,
                "max_tokens": 2200
            }

            response = requests.post(self.base_url, headers=self.headers, json=payload)
            response.raise_for_status()

            result = response.json()
            message = result['choices'][0]['message']
            content = message.get('content')
            if not content and 'reasoning' in message:
                content = message['reasoning']
            return content or ""

        except requests.exceptions.HTTPError as e:
            return f"Groq API HTTP Error: {e.response.status_code} - {e.response.text}"
        except Exception as e:
            return f"Error communicating with LLM: {str(e)}"


# Initialize LLM
if GROQ_API_KEY:
    llm = GroqLLM(GROQ_API_KEY)
    print("✅ Groq LLM initialized successfully")
else:
    print("❌ Cannot initialize LLM without an API key")


✅ Groq API key loaded successfully
🚀 Environment setup complete!
✅ Groq LLM initialized successfully


DUAL LAYER MEMORY SYSTEM

- Short term memory for temporal tracking of code flow and memory
- Long term memory to learn from past conversations

 It demonstrates how agentic systems maintain state across interactions and learn from previous coding interactions.

In [40]:
# Base Memory System

class CodingMemory:
    """Memory system for the coding assistant"""

    def __init__(self):
        # Short-term memory (current session)
        self.session_context = {
            "current_task": None,
            "generated_code": [],
            "execution_results": [],
            "errors_encountered": [],
            "variables_in_scope": {}
        }

        #long term memory system

        self.coding_history = {
            "successful_patterns": [],
            "frequent_errors": [],
            "user_preferences": {
                "coding_style": "pythonic",
                "preferred_libraries": ["numpy", "pandas", "requests"],
                "difficulty_level": "intermediate"

            },
            "session_count": 0
        }

    def update_session_context(self, key:str, value: Any):
        """update short term memory"""

        self.session_context[key] = value
        print(f"Updated session context: {key}")

    def add_to_history(self, category: str, item: Dict):
        """Add to long-term memory"""
        if category in self.coding_history:
            self.coding_history[category].append({
                **item,
                "timestamp": datetime.now().isoformat()
            })
        print(f"Added to coding history: {category}")


    def get_context_summary(self) -> str:
        """Get current context for LLM"""

        return f"""
Current Session Context:
- Task: {self.session_context.get('current_task', 'None')}
- Generated Code Blocks: {len(self.session_context['generated_code'])}
- Execution Results: {len(self.session_context['execution_results'])}
- Recent Errors: {len(self.session_context['errors_encountered'])}

User Preferences:
- Coding Style: {self.coding_history['user_preferences']['coding_style']}
- Preferred Libraries: {', '.join(self.coding_history['user_preferences']['preferred_libraries'])}
- Difficulty Level: {self.coding_history['user_preferences']['difficulty_level']}
"""

# Initialize memory system
memory = CodingMemory()
print("🧠 Memory system initialized")
print("Current context:", memory.get_context_summary())


🧠 Memory system initialized
Current context: 
Current Session Context:
- Task: None
- Generated Code Blocks: 0
- Execution Results: 0
- Recent Errors: 0

User Preferences:
- Coding Style: pythonic
- Preferred Libraries: numpy, pandas, requests
- Difficulty Level: intermediate



CODE GENERATOR --

A comprehensive code generator tool which utilizes user preferences, updates memory with artifacts based on user interactions, system prompt with context retrieval, clean system output.

In [41]:
class CodeGenerator:
    """Base core generator tool"""
    def __init__(self, llm: GroqLLM, memory: CodingMemory):
        self.llm = llm
        self.memory = memory
        self.name = "code_generator"

    def generate_code(self, description:str, language: str = "python") -> Dict[str, Any]:
        """Generate code based on Natural Language description"""

        #get context from the base memory

        context = self.memory.get_context_summary()

        #create system prompt

        messages = [
            {
                "role": "system",
                "content": f"""You are an expert {language} programmer. Generate clean, well-commented code based on the user's description.

                Context from previous session:
                {context}

                Requirements:
                1. Write clean, readable code
                2. Add helpful comments
                3. Follow best practices
                4. Include error handling where appropriate
                5. Return only the code, no explanations
                """
            },

            #user prompt
            {
                "role": "user",
                "content": f"Generate {language} code for {description}"
            }

        ]

        try:
            generated_code = self.llm.generate_response(messages)

            # Clean up the code (remove Markdown formatting if present)
            if "```" in generated_code:
                lines = generated_code.split('\n')
                code_lines = []
                in_code_block = False

                for line in lines:
                    if line.strip().startswith('```'):
                        in_code_block = not in_code_block
                        continue
                    if in_code_block:
                        code_lines.append(line)

                generated_code = '\n'.join(code_lines)


            #update memory
            self.memory.update_session_context("current_task", description)
            self.memory.session_context['generated_code'].append({
                "description": description,
                "code": generated_code,
                "language": language,
                "timestamp": datetime.now().isoformat()

            })

            result = {
                "success": True,
                "code": generated_code,
                "description": description,
                "language": language
            }

            print(f"✅ Code generated successfully for description: {description}")
            return result

        except Exception as e:
            error_result = {
                "success": False,
                "error": str(e),
                "description": description
            }
            print(f"❌ Code generation failed: {str(e)}")
            return error_result

# Initialize code generator
if 'llm' in globals() and GROQ_API_KEY:
    code_generator = CodeGenerator(llm, memory)
    print("🔧 Code Generator Tool initialized")


🔧 Code Generator Tool initialized


CODING LINTER TOOL --

Using Python Abstract Syntax Tree to check syntax style, applying linting and update the memory according to coding style.

In [42]:
# Code Linter Tool

class LinterTool:
    """Tool for checking code syntax and style"""

    def __init__(self, memory: CodingMemory):
        self.memory = memory
        self.name = "code_linter"

    def lint_code(self, code:str, language:str = "python") -> Dict[str, Any]:
        """Lint code for syntax errors and type issues"""

        issues = []
        syntax_valid = True

        if language.lower() == "python":
            try:
                ast.parse(code)
                syntax_valid = True
            except SyntaxError as e:
                syntax_valid = False
                issues.append({
                    "severity": "error",
                    "type": "syntax_error",
                    "line": e.lineno,
                    "message": str(e)

                })

            # Basic style checks
            lines = code.split('\n')
            for i, line in enumerate(lines, 1):
                # Check line length
                if len(line) > 100:
                    issues.append({
                        "type": "style",
                        "line": i,
                        "message": f"Line too long ({len(line)} characters)",
                        "severity": "warning"
                    })

                #check for missing docstrings in functions
                if line.strip().startswith('def ') and i < len(lines):
                    next_line = lines[i].strip() if i < len(lines) else ""
                    if not next_line.startswith('"""') and not next_line.startswith("'''"):
                        issues.append({
                            "type": "style",
                            "line": i,
                            "message": "Missing Docstring values",
                            "severity": "info"
                        })


        # Update memory with linting results
        lint_result = {
            "syntax_valid": syntax_valid if language.lower() == "python" else True,
            "issues": issues,
            "total_issues": len(issues),
            "code": code
        }

        self.memory.session_context["errors_encountered"].extend(issues)

        #if issues found
        if issues:
            print(f" Found {len(issues)} linting issues")
            for issue in issues:
                print(f" Line {issue['line']} : {issue['message']} ({issue['severity']})")

        else:
            print("Code passed linting checks.")

        return lint_result


#initialize linter tool
code_lint = LinterTool(memory)
print("Code Linter Initialized")


Code Linter Initialized


CODE EXECUTOR TOOL --

EXECUTES CODE SNIPPETS IN A SANDBOXED ENVIRONMENT, APPLIES NAMESPACE FOR RESTRICTED CODE EXECUTION AND PREVENT CROSS-EXECUTION. STRUCTURIZES OUTPUT AND GIVES CLEAR FEEDBACK.

In [43]:
#Code Executor Tool
class CodeExecutorTool:

    def __init__(self, memory: CodingMemory):
        self.memory = memory
        self.namespace_sandbox = {} #unique namespace for isolated code execution
        self.name = "code_executor"


    def execute_code(self, code: str, language: str = "python") -> Dict[str, Any]:
        """Execute code safely in a sandboxed environment"""

        if language.lower() != "python":
            return {
                "success": False,
                "error": f"Execution not supported for language {language}",
                "output": "",
                "execution_time": 0
            }

        #capture stdout and stderr
        old_stdout = sys.stdout
        old_stderr = sys.stderr

        stdout_capture = StringIO()
        stderr_capture = StringIO()


        start_time = datetime.now()

        #redirect stdout and stderr

        try:
            sys.stdout = stdout_capture
            sys.stderr = stderr_capture

            #execute code in sandbox
            exec(code, self.namespace_sandbox)

            #calculate execution time
            execution_time  = (datetime.now() - start_time).total_seconds()

            #get captured value
            output = stdout_capture.getvalue()
            err_output = stderr_capture.getvalue()

            result = {
                "success": True,
                "output": output,
                "error_output": err_output,
                "execution_time": execution_time,
                "variables_created": (self.namespace_sandbox.keys())

            }

            # Update memory
            self.memory.session_context["execution_results"].append({
                "code": code,
                "output": output,
                "success": True,
                "timestamp": datetime.now().isoformat()
            })

            self.memory.session_context["variables_in_scope"] = {
                k: str(type(v).__name__) for k, v in self.namespace_sandbox.items()
                if not k.startswith('__')
            }

            print(f"✅ Code executed successfully in {execution_time:.3f}s")
            if output:
                print("📤 Output:", output.strip())

            return result

        except Exception as e:
            execution_time = (datetime.now() - start_time).total_seconds()

            result = {
                "success": False,
                "error": str(e),
                "output": stdout_capture.getvalue(),
                "error_output": stderr_capture.getvalue(),
                "execution_time": execution_time
            }

        # Update memory with error
            self.memory.session_context["execution_results"].append({
                "code": code,
                "error": str(e),
                "success": False,
                "timestamp": datetime.now().isoformat()
            })

            print(f"❌ Code execution failed: {str(e)}")
            return result

        #restore stdout and stderr
        finally:
            sys.stdout = old_stdout
            sys.stderr = old_stderr

# Initialize executor
code_executor = CodeExecutorTool(memory)
print("⚡ Code Executor Tool has been initialized")

⚡ Code Executor Tool has been initialized


!!! Why we need stdout and stderr


Separation of Results from Logs


In a sandbox, the surrounding system often needs to parse or save the output generated by the executed code.

stdout is reserved for expected program data (such as JSON, calculated values, or user-facing text).

stderr is used for diagnostic messages, warnings, execution traces, or system-level debugging info.

If both streams were merged into a single channel, debugging logs or warning messages could corrupt structured output data (e.g., throwing a SyntaxError when attempting to parse a JSON payload).

ReAct Reasoning Engine ---

Reasoning -- Action -- Execute > Loop


this helps to refine the output for accurate results, checks if max iterations have been reached and applies reasoning strategies. Acts, and plans execution.

Initiates different tools for executing different tasks, records output and displays results and feedback

In [44]:
#ReAct Reasoning Tool

class ReActCodingAgent:
    """ReAct based reasoning engine for coding engine
    Reasoning: Thinking about what needs to be done next
    Action: Choosing which tool to use
    Observation: Processing the results of tool execution

    """

    def __init__(self, llm: GroqLLM, memory: CodingMemory, tools: Dict[str, Any], max_iterations: int = 5):
        self.llm = llm
        self.memory = memory
        self.tools = tools
        self.max_iterations = max_iterations


    def reason_and_act(self, user_request: str) -> Dict[str, Any]:
        """Main reasoning and action loop"""

        print(f"\n🤖 Processing request: {user_request}")
        print("=" * 50)

        reasoning_trace = []
        final_response = None

        for iteration in range(self.max_iterations):
            print(f"\n🔄 ReAct Iteration {iteration + 1}")


            #THOUGHT: -- analyze user context
            thought = self._generate_thought(user_request, reasoning_trace)
            print(f"💭 Thought: {thought}")
            reasoning_trace.append({"type": "thought", "content": thought})

            #ACTION: -- decide tools and execution plan
            action_planning = self._plan_action(user_request, thought, reasoning_trace)
            print(f"🎯 Action Plan: {action_planning}")

            if action_planning.get("action") == "complete":
                final_response = action_planning
                break

            #EXECUTE: -- execute and observe planned action
            observation = self._execute_action(action_planning)
            print(f"Observation: {observation['summary']}")
            reasoning_trace.append({"type": "observation", "content": observation})

            # Check if we should continue
            if observation.get("task_complete", False):
                final_response = observation
                break

        # Generate final response
        if not final_response:
            final_response = {
                "success": True if self.memory.session_context["generated_code"] else False,
                "message": "Task complete"
            }


        return {
            "success": final_response.get("success", True),
            "result": final_response,
            "reasoning_trace": reasoning_trace,
            "iterations": len([t for t in reasoning_trace if t["type"] == "thought"])
        }

    def _generate_thought(self, user_request:str, trace: List[Dict]) -> str:
        """Generate reasoning thought based on current context"""

        context = self.memory.get_context_summary()
        trace_summary = self._summarize_trace(trace)

        messages = [
            {
                "role": "system",
                "content": """You are a coding assistant's reasoning engine. Analyze the situation and generate a clear thought about what needs to be done next.
                Be professional in your execution and provide concise answers.

Consider:
1. What is the user asking for?
2. What has been done so far?
3. What capabilities are available?
4. What should be the next step?

Available capabilities: code_generator, code_linter, code_executor

If code has already been generated and no errors remain, suggest completing the task.
Respond with a single, clear thought about the next step."""
            },
            {
                "role": "user",
                "content": f"""
User Request: {user_request}

Current Context:
{context}

Previous Actions:
{trace_summary}

What should I think about next?
"""
            }
        ]


        #generate response
        return self.llm.generate_response(messages=messages, temperature=0.1)

    def _plan_action(self, user_request: str, thought: str, trace: List[Dict]) -> Dict[str, Any]:
        """Plan the next action based on current thought"""

        trace_summary = self._summarize_trace(trace)

        messages = [
            {
                "role": "system",
                "content": """You are a coding assistant's action planner. Based on the current thought and previous actions, decide what action to take.

Available actions:
1. generate_code: Create new code from description
2. lint_code: Check existing code for issues
3. execute_code: Run code and see results
4. complete: Task is finished

Respond with JSON format:
{
    "action": "action_name",
    "parameters": {"param1": "value1"},
    "reasoning": "why this action"
}

If code has already been generated successfully, select "complete"."""
            },
            {
                "role": "user",
                "content": f"""
User Request: {user_request}
Current Thought: {thought}
Previous Actions:
{trace_summary}

What action should I take next?
"""
            }
        ]

        response = self.llm.generate_response(messages, temperature=0.1)


        try:
            #try to parse JSON response
            if "{" in response and "}" in response:
                start = response.find("{")
                end = response.rfind("}") + 1
                json_str = response[start:end]
                data = json.loads(json_str)
                if isinstance(data, dict) and "action" in data:
                    return data

            if "complete" in response.lower() or "finish" in response.lower():
                return {"action": "complete", "parameters": {}}
            elif "lint" in response.lower():
                return {"action": "lint_code", "parameters": {}}
            elif "execute" in response.lower() or "run" in response.lower():
                return {"action": "execute_code", "parameters": {}}
            else:
                return {"action": "generate_code", "parameters": {"description": user_request}}
        except:
            # Default action if parsing fails
            return {"action": "generate_code", "parameters": {"description": user_request}}


    def _execute_action(self, action_plan: Dict[str, Any]) -> Dict[str, Any]:
        """Execute planned action using appropriate tools"""

        action = action_plan.get("action", "generate_code")
        params = action_plan.get("parameters", {})

        if action == "generate_code":
            description = params.get("description", "Generate code")
            result = self.tools["code_generator"].generate_code(description)

            return {
                "success": result["success"],
                "summary": f"Generated code for: {description}",
                "details": result,
                "task_complete": False
            }

        elif action == "lint_code":
            # Get the most recent generated code
            recent_code = self.memory.session_context["generated_code"]
            if recent_code:
                code = recent_code[-1]["code"]
                result = self.tools["code_linter"].lint_code(code)

                return {
                    "success": result["syntax_valid"],
                    "summary": f"Linted code - found {result['total_issues']} issues",
                    "details": result,
                    "task_complete": result["total_issues"] == 0
                }
            else:
                return {
                    "success": False,
                    "summary": "No code available to lint",
                    "task_complete": False
                }

        elif action == "execute_code":
            # Get the most recent generated code
            recent_code = self.memory.session_context["generated_code"]
            if recent_code:
                code = recent_code[-1]["code"]
                result = self.tools["code_executor"].execute_code(code)

                return {
                    "success": result["success"],
                    "summary": f"Executed code - {'success' if result['success'] else 'failed'}",
                    "details": result,
                    "task_complete": result["success"]
                }
            else:
                return {
                    "success": False,
                    "summary": "No code available to execute",
                    "task_complete": False
                }

        else:
            return {
                "success": True,
                "summary": "Task completed",
                "task_complete": True
            }

    def _summarize_trace(self, trace: List[Dict]) -> str:
        """Summarize the reasoning trace for context"""
        if not trace:
            return "No previous actions"

        summary = []
        for i, step in enumerate(trace[-3:], 1):  # Last 3 steps
            if step["type"] == "thought":
                summary.append(f"{i}. Thought: {step['content'][:100]}...")
            elif step["type"] == "observation":
                summary.append(f"{i}. Result: {step['content']['summary']}")

        return "\n".join(summary)

# Initialize ReAct agent
tools_dict = {
    "code_generator": code_generator,
    "code_linter": code_lint,
    "code_executor": code_executor
}

if GROQ_API_KEY and 'llm' in globals():
    react_agent = ReActCodingAgent(llm, memory, tools_dict)
    print("🧠 ReAct Coding Agent initialized")
else:
    print("❌ Cannot initialize ReAct agent without API key")


🧠 ReAct Coding Agent initialized


In [45]:
# Demo 1 - Simple Code Generation

print("🎯 DEMO 1: Simple Code Generation")
print("=" * 40)

if GROQ_API_KEY:
    # Simple request for code generation
    result1 = react_agent.reason_and_act(
        "Create a Python function that calculates the palindrome of a number"
    )

    print(f"\n📊 Result Summary:")
    print(f"Success: {result1['success']}")
    print(f"Iterations: {result1['iterations']}")

    # Show the generated code
    if memory.session_context["generated_code"]:
        latest_code = memory.session_context["generated_code"][-1]
        print(f"\n📝 Generated Code:")
        print("-" * 30)
        print(latest_code["code"])
        print("-" * 30)
else:
    print("⚠️  Skipping demo - API key required")

🎯 DEMO 1: Simple Code Generation

🤖 Processing request: Create a Python function that calculates the palindrome of a number

🔄 ReAct Iteration 1
💭 Thought: Ask the user to clarify whether they want a function that checks if a number is a palindrome, reverses its digits, or finds the next palindrome greater than the given number.
🎯 Action Plan: {'action': 'generate_code', 'parameters': {'description': 'Python function that returns the next palindrome greater than a given integer. Includes helper functions to check for palindrome and to reverse digits.'}, 'reasoning': 'The user asked for a function that calculates the palindrome of a number. Interpreting this as finding the next palindrome greater than the input, we provide a clear implementation with documentation and tests.'}
Updated session context: current_task
✅ Code generated successfully for description: Python function that returns the next palindrome greater than a given integer. Includes helper functions to check for palindrome

In [46]:
# Demo 2 - Complex Task with Tool Chaining

print("\n🎯 DEMO 2: Complex Task with Tool Chaining")
print("=" * 45)

if GROQ_API_KEY:
    # More complex request that requires multiple tools
    result2 = react_agent.reason_and_act(
        "Create a Python function that reads a CSV file and calculates statistics, then test it with sample data"
    )

    print(f"\n📊 Result Summary:")
    print(f"Success: {result2['success']}")
    print(f"Iterations: {result2['iterations']}")

    # Show execution results
    if memory.session_context["execution_results"]:
        latest_execution = memory.session_context["execution_results"][-1]
        if latest_execution["success"]:
            print(f"\n✅ Execution Output:")
            print(latest_execution.get("output", "No output"))
        else:
            print(f"\n❌ Execution Error:")
            print(latest_execution.get("error", "Unknown error"))
else:
    print("⚠️  Skipping demo - API key required")


🎯 DEMO 2: Complex Task with Tool Chaining

🤖 Processing request: Create a Python function that reads a CSV file and calculates statistics, then test it with sample data

🔄 ReAct Iteration 1
💭 Thought: Generate a concise, pythonic function that uses pandas to read a CSV file, compute basic statistics (mean, median, std, min, max) for numeric columns, and then write a small test block that creates a sample CSV in memory, calls the function, and prints the results.
🎯 Action Plan: {'action': 'generate_code', 'parameters': {'description': 'Create a Python function that reads a CSV file and calculates statistics (mean, median, std, min, max) for numeric columns, then test it with sample data.'}, 'reasoning': 'The user explicitly requests a function and a test block. We need to provide the code.'}
Updated session context: current_task
✅ Code generated successfully for description: Create a Python function that reads a CSV file and calculates statistics (mean, median, std, min, max) for numer

In [47]:
# Performance Metrics and Analysis

class AgentAnalyzer:
    """Analyze the performance and behavior of our coding assistant"""

    def __init__(self, memory: CodingMemory):
        self.memory = memory

    def generate_performance_report(self) -> Dict[str, Any]:
        """Generate comprehensive performance analysis"""

        # Calculate success rates
        total_generations = len(self.memory.session_context["generated_code"])
        total_executions = len(self.memory.session_context["execution_results"])
        successful_executions = sum(1 for r in self.memory.session_context["execution_results"] if r["success"])

        success_rate = (successful_executions / total_executions * 100) if total_executions > 0 else 0

        # Analyze error patterns
        error_types = {}
        for error in self.memory.session_context["errors_encountered"]:
            error_type = error.get("type", "unknown")
            error_types[error_type] = error_types.get(error_type, 0) + 1

        # Calculate average execution time
        execution_times = []
        for result in self.memory.session_context["execution_results"]:
            if "execution_time" in result:
                execution_times.append(result["execution_time"])

        avg_execution_time = sum(execution_times) / len(execution_times) if execution_times else 0

        return {
            "session_stats": {
                "total_code_generations": total_generations,
                "total_executions": total_executions,
                "successful_executions": successful_executions,
                "success_rate_percent": round(success_rate, 2),
                "average_execution_time": round(avg_execution_time, 3)
            },
            "error_analysis": {
                "total_errors": len(self.memory.session_context["errors_encountered"]),
                "error_types": error_types
            },
            "memory_usage": {
                "variables_in_scope": len(self.memory.session_context["variables_in_scope"]),
                "context_size": len(str(self.memory.session_context))
            }
        }

    def print_report(self):
        """Print formatted performance report"""
        report = self.generate_performance_report()

        print("📊 AGENT PERFORMANCE REPORT")
        print("=" * 40)

        print("\n📈 Session Statistics:")
        stats = report["session_stats"]
        for key, value in stats.items():
            print(f"  - {key.replace('_', ' ').title()}: {value}")

        print("\n🐛 Error Analysis:")
        error_stats = report["error_analysis"]
        print(f"  - Total Errors: {error_stats['total_errors']}")
        if error_stats["error_types"]:
            print("  - Error Types:")
            for error_type, count in error_stats["error_types"].items():
                print(f"    • {error_type}: {count}")

        print("\n💾 Memory Usage:")
        memory_stats = report["memory_usage"]
        for key, value in memory_stats.items():
            print(f"  - {key.replace('_', ' ').title()}: {value}")

# Generate performance analysis
analyzer = AgentAnalyzer(memory)
analyzer.print_report()

📊 AGENT PERFORMANCE REPORT

📈 Session Statistics:
  - Total Code Generations: 8
  - Total Executions: 0
  - Successful Executions: 0
  - Success Rate Percent: 0
  - Average Execution Time: 0

🐛 Error Analysis:
  - Total Errors: 0

💾 Memory Usage:
  - Variables In Scope: 0
  - Context Size: 15174


In [49]:
# Security Demonstration

class SecurityValidator:
    """Demonstrate security features of the coding assistant"""

    def __init__(self, executor: CodeExecutorTool):
        self.executor = executor

    def test_security_features(self):
        """Test various security scenarios"""

        print("🔒 SECURITY TESTING")
        print("=" * 30)

        # Test 1: Syntax error handling
        print("\n🧪 Test 1: Syntax Error Handling")
        malformed_code = "def broken_function(\n    print('missing closing parenthesis'"
        result = self.executor.execute_code(malformed_code)
        print(f"Result: {'✅ Handled safely' if not result['success'] else '❌ Should have failed'}")

        # Test 2: Runtime error handling
        print("\n🧪 Test 2: Runtime Error Handling")
        runtime_error_code = """
def divide_by_zero():
    return 10 / 0

result = divide_by_zero()
"""
        result = self.executor.execute_code(runtime_error_code)
        print(f"Result: {'✅ Handled safely' if not result['success'] else '❌ Should have failed'}")

        # Test 3: Safe execution with output
        print("\n🧪 Test 3: Safe Execution")
        safe_code = """
def safe_function():
    numbers = [1, 2, 3, 4, 5]
    return sum(numbers)

result = safe_function()
print(f"Sum of numbers: {result}")
"""
        result = self.executor.execute_code(safe_code)
        print(f"Result: {'✅ Executed safely' if result['success'] else '❌ Should have succeeded'}")
        if result['success']:
            print(f"Output: {result['output'].strip()}")

        # Test 4: Namespace isolation
        print("\n🧪 Test 4: Namespace Isolation")
        isolation_test = """
# This should not affect the global namespace
import sys
test_variable = "isolated"
print(f"Test variable: {test_variable}")
"""
        result = self.executor.execute_code(isolation_test)
        print(f"Result: {'✅ Properly isolated' if result['success'] else '❌ Isolation failed'}")

# Run security tests
security_validator = SecurityValidator(code_executor)
security_validator.test_security_features()

🔒 SECURITY TESTING

🧪 Test 1: Syntax Error Handling
Result: ✅ Handled safely

🧪 Test 2: Runtime Error Handling
Result: ✅ Handled safely

🧪 Test 3: Safe Execution
Result: ✅ Executed safely
Output: Sum of numbers: 15

🧪 Test 4: Namespace Isolation
Result: ✅ Properly isolated
